In [1]:
from crewai import Agent, Task, Crew, LLM
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
import os
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key, get_exa_api_key
from IPython.display import Markdown
import yaml

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/firecrawl/firecrawl.py:78: UserWarning: Field name "json" in "ChangeTrackingData" shadows an attribute in parent "BaseModel"
  class ChangeTrackingData(pydantic.BaseModel):


In [2]:
# import packages needed for the custom tool
from crewai.tools import BaseTool
from crewai import LLM
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json

# Define the custom tool for creating plots
class CustomPlotTool(BaseTool):
    ### START CODE HERE ###
    name: str = "Create custom plots"
    description: str = ("This a tool for automatically creating custom plots based on a research result. "
                        "This tools automatically generates the plots from a text input, which should have fact checked information. "
                        "Pass the full validated information gathered so far as a string."
                        )
    def _run(self, research: str) -> str:
    ### END CODE HERE ###
        try:
            extraction_prompt = f"""
            You are an expert data visualization assistant. Analyze the provided research text and identify meaningful, insightful charts that can be created to visualize quantifiable data supporting the research's key insights and findings. Only suggest charts for data that includes numerical values, measurable trends, comparisons, or categorical distributions that can be effectively plotted.

            Focus on creating visualizations that highlight trends, comparisons, distributions, or relationships that add value to the research. Avoid suggesting charts for purely qualitative or non-quantifiable information.

            For each chart, provide a JSON object with:
              - "chart_type" (string: choose from "line" for trends over time/continuous, "bar" for comparisons, "histogram" for distributions, "scatter" for relationships, "pie" for proportions)
              - "x_axis" (string: variable name for x-axis, e.g., "year", "category")
              - "y_axis" (string: variable name for y-axis, e.g., "value", "count")
              - "color" (string: optional variable for color grouping/hue, or null if not applicable)
              - "Title" (string: descriptive, insightful title that explains what the chart shows)
              - "data" (dictionary: keys matching x_axis, y_axis, and color variables; values as lists of extracted numerical/categorical data from the research)

            Ensure data is accurately extracted and formatted as lists. If a variable has multiple series (e.g., for color), include all in the data dictionary.

            If no quantifiable data suitable for meaningful visualization is present in the research, return an empty array [].

            Text:
            {research}

            Example output (return valid JSON only):
            [
              {{"chart_type": "line", "x_axis": "year", "y_axis": "funding_amount", "color": "sector", "Title": "AI Research Funding Trends by Sector", "data": {{"year": [2020, 2021, 2022], "funding_amount": [2.5, 3.8, 5.2], "sector": ["Healthcare", "Finance", "Tech"]}}}},
              {{"chart_type": "bar", "x_axis": "tool_name", "y_axis": "adoption_rate", "color": null, "Title": "Market Adoption Rates of AI Tools", "data": {{"tool_name": ["ToolA", "ToolB", "ToolC"], "adoption_rate": [45, 67, 23]}}}}
            ]

            Return only the JSON array, no additional text or explanations.
            """
            llm = LLM(model="gpt-4o-mini",)  # Initialize the LLM instance
            llm_response = llm.call([{"role": "user", "content": extraction_prompt}])

            # Clean the response to extract just the JSON part
            llm_response = llm_response.strip()
            if llm_response.startswith('```json'):
                llm_response = llm_response[7:]  # Remove ```json
            if llm_response.endswith('```'):
                llm_response = llm_response[:-3]  # Remove ```
            llm_response = llm_response.strip()

            # --- Step 2: Parse the LLM output ---
            charts_data = json.loads(llm_response)

            if not isinstance(charts_data, list) or len(charts_data) == 0:
                return "No information found in the research to visualize."

            plots_created = []

            # --- Step 3: Create plots for each chart ---
            for i, chart_info in enumerate(charts_data):
                try:
                    # Extract chart configuration
                    chart_type = chart_info.get("chart_type", None).lower()
                    x_axis = chart_info.get("x_axis", "x")
                    y_axis = chart_info.get("y_axis", "y") 
                    title = chart_info.get("Title", f"Chart {i+1}")
                    hue = chart_info.get("color", None)
                    data = chart_info.get("data", {})

                    # Create DataFrame from the data
                    df = pd.DataFrame(data)

                    if df.empty:
                        continue

                    # Create the plot
                    plt.figure(figsize=(10, 6))

                    if chart_type == "line":
                        sns.lineplot(data=df, x=x_axis, y=y_axis, marker="o", hue=hue)
                    elif chart_type in ["bar", "column"]:
                        sns.barplot(data=df, x=x_axis, y=y_axis, hue=hue)
                    elif chart_type == "histogram":
                        plt.hist(df[y_axis], bins=10, alpha=0.7, hue=hue)
                        plt.xlabel(y_axis)
                        plt.ylabel("Frequency")
                    elif chart_type == "scatter":
                        # Default to scatter plot
                        sns.scatterplot(data=df, x=x_axis, y=y_axis, hue=hue)
                    elif chart_type == "pie":
                        # For pie chart, assume y_axis is values, x_axis is labels
                        plt.pie(df[y_axis], labels=df[x_axis], autopct='%1.1f%%', startangle=90)
                        plt.title(title)
                        plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.

                    plt.title(title)
                    plt.xticks(rotation=45)
                    plt.tight_layout()

                    # --- Step 4: Save the plot ---
                    os.makedirs("plots", exist_ok=True)
                    filename = f"plots/plot_{i+1}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
                    plt.savefig(filename, dpi=300, bbox_inches='tight')
                    plt.close()

                    plots_created.append(filename)

                except Exception as e:
                    print(f"Error creating chart {i+1}: {str(e)}")
                    continue

            if plots_created:
                return f"Successfully created {len(plots_created)} plots: {', '.join(plots_created)}"
            else:
                return "No plots could be created from the extracted data."

        except json.JSONDecodeError as e:
            return f"Error parsing LLM response as JSON: {str(e)}"
        except Exception as e:
            return f"Error generating smart plot: {str(e)}"

In [3]:
# create tools instances
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))
scrape_website_tool = ScrapeWebsiteTool()

/var/folders/hf/81yfsz694rx2svmh31g42bb80000gn/T/ipykernel_14060/2777457341.py:2: DeprecationWarning: EXASearchTool is deprecated and will be removed in a future release; use ExaSearchTool instead.
  exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))


In [4]:
# load the configuration file for the agents
with open('config/agents.yaml', 'r') as file:
        agent_config = yaml.safe_load(file)


# create the agents using the configuration
research_planner = Agent(
        config=agent_config['research_planner'],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )

internet_researcher = Agent( 
        config=agent_config['internet_researcher'],
        verbose=True,
        tools=[exa_search_tool, scrape_website_tool],
        max_rpm=150,
        max_iter=15
        )

fact_checker = Agent(
        config=agent_config['fact_checker'],
        verbose=True,
        tools=[exa_search_tool, scrape_website_tool],
        max_rpm=150,
        max_iter=15
        )

report_writer = Agent(
        config=agent_config['report_writer'],
        verbose=True,
        ### START CODE HERE ### 
        # add the automatic plot tool
        tools=[CustomPlotTool()],
        ### END CODE HERE ###
        max_rpm=150,
        max_iter=15
        )

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/agent/core.py:358: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/agent/core.py:366: DeprecationWarning: deprecated
  if self.allow_code_execution:
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/agent/core.py:377: DeprecationWarning: deprecated
  if self.reasoning and self.planning_config is None:


In [5]:
import re

# write the custom guardrail function
def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw 
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )
    
    # convert the output to lowercase
    output_lower = output.lower()

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):
        return (False, 
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False, 
                "The report must include an Insights section with a header like '## Insights'"
                )

    # check that the citations (or references) section exists
    if not re.search(r'#+.*citations|#+.*references', output_lower): 
        return (False, 
                "The report must include a Citations (or References) section with a header like '## Citations'"
                )
    return (True, output)

In [6]:
# load the configuration file for the tasks
with open('config/tasks.yaml', 'r') as file:
    task_config = yaml.safe_load(file)


# create the tasks using the configuration
create_research_plan = Task( 
    config=task_config['create_research_plan'],
    agent=research_planner 
)

gather_research_data = Task(
    config=task_config['gather_research_data'],
    agent=internet_researcher,
)

verify_information_quality = Task(
    config=task_config['verify_information_quality'],
    agent=fact_checker, 
)

write_final_report = Task( 
    config=task_config['write_final_report'],
    agent=report_writer, 
    guardrails=[write_report_guardrail],
)

In [7]:
def save_file_hook(result):
    """
    Save the final research report to a local markdown file
    """
    try:
        # Get the final report content from the last task output
        if hasattr(result, 'tasks_output') and result.tasks_output:
            report_content = result.tasks_output[-1].raw
        else:
            report_content = str(result)
        
        filename = f"research_report-p2.md"
        
        # Save to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report successfully saved to: {filename}")
        
    except Exception as e:
        print(f"Error saving report to file: {str(e)}")

In [8]:
# Create the urban planning crew
deep_research_crew = Crew(
    # include all the agents
    agents=[research_planner, 
            internet_researcher, 
            fact_checker, 
            report_writer],
    # include all the tasks in the order to be executed
    tasks=[create_research_plan, 
           gather_research_data, 
           verify_information_quality, 
           write_final_report],
    # add memory to the crew
    memory=True,
    # add the after kickoff hook
    after_kickoff_callbacks=[save_file_hook]
)

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/crew.py:635: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(self.function_calling_llm, LLM):
/opt/homebrew/Cellar/python@3.13/3.13.3/Frameworks/Python.framework/Versions/3.13/lib/python3.13/contextlib.py:136: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x128433100>
  def __enter__(self):


In [9]:
### START CODE HERE ###

# write your query in the "user_query" value
inputs = {
        "user_query": "Evaluate the top five emerging AI tools for automating competitive market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing firm."
}
### END CODE HERE ###   

In [10]:
# Execute the crew's tasks

import asyncio

# Recommended way in notebooks
result = await deep_research_crew.kickoff_async(inputs=inputs)

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/telemetry/telemetry.py:431: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if getattr(agent, "function_calling_llm", None)
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/telemetry/telemetry.py:438: DeprecationWarning: deprecated
  "allow_code_execution?": getattr(
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/crews/utils.py:362: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  crew.function_calling_llm,
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/crews/utils.py:104: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if not agent

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Break down the research query "Evaluate the top five emerging AI tools for automating competitive        │
│  market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing   │
│  firm." into specific topics and key questions that need investigation. Create a focused research plan.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/types.py:364: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  age_seconds = (datetime.utcnow() - record.created_at).total_seconds()
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/storage/lancedb_storage.py:352: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().isoformat()
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/types.py:364: DeprecationWarning: datetime.datetime.utcnow() is deprecated 

Tool search_memory executed with result: Found memories:
- (score=0.76) Klue's main features include integrations with CRM and Slack, real-time competitor updates, and automatic battlecards.
  categories: competitive analysis, tools, SaaS Pr...
Tool search_memory executed with result: Found memories:
- (score=0.78) Identified tools include Klue, Crayon, Semrush, Similarweb, and Competely.
  categories: market_research, competitive_analysis, digital_marketing
  entities: ['Klue', 'C...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Research Plan to Evaluate Top Five Emerging AI Tools for Automating Competitive Market Analysis             │
│                                                                                                                 │
│  ### 1) Main Research Topics to Investigate                                                                     │
│  - Identification of AI Tools                                                                                   │
│  - Features of Each Tool                                                                                        │
│  - Limitations of Each Tool                                                                                     │
│  - Cost Structure and Pricing Models                                                                            │
│  - Ideal Use Cases for Mid-Sized Marketing Firms                                                                │
│                                                                                                                 │
│  ### 2) Key Questions for Each Topic                                                                            │
│                                                                                                                 │
│  #### Identification of AI Tools                                                                                │
│  - What are the top five emerging AI tools for automating competitive market analysis?                          │
│  - How do these tools differ in their approach to competitive analysis?                                         │
│                                                                                                                 │
│  #### Features of Each Tool                                                                                     │
│  - What are the unique features of each tool (Klue, Crayon, Semrush, Similarweb, Competely)?                    │
│    - For Klue: What integrations and functionalities does it offer (e.g., CRM, Slack, real-time updates)?       │
│    - For Crayon: What specific capabilities in website monitoring and report generation does it have?           │
│    - For Semrush: What SEO features does it emphasize (e.g., keyword gap analysis, traffic analytics)?          │
│    - For Similarweb: What insights does it provide on digital market share and website traffic?                 │
│    - For Competely: How does it facilitate quick competitor reports and rapid analysis?                         │
│                                                                                                                 │
│  #### Limitations of Each Tool                                                                                  │
│  - What common limitations do these tools have regarding data accuracy, integration, and usability?             │
│  - Are there any specific drawbacks unique to each tool that may affect their effectiveness for mid-sized       │
│  firms?                                                                                                         │
│                                                                                                                 │
│  #### Cost Structure and Pricing Models                                                                         │
│  - What is the pricing structure for each tool?        

...

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/types.py:364: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  age_seconds = (datetime.utcnow() - record.created_at).total_seconds()
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/storage/lancedb_storage.py:352: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().isoformat()
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/types.py:364: DeprecationWarning: datetime.datetime.utcnow() is deprecated 

Tool search_memory executed with result: Found memories:
- (score=0.77) Klue's main features include integrations with CRM and Slack, real-time competitor updates, and automatic battlecards.
  categories: competitive analysis, tools, SaaS Pr...
Tool search_memory executed with result (from cache): Found memories:
- (score=0.81) Klue's pricing ranges from $150 to $1,000 per month, with higher tiers potentially incurring consulting fees.
  categories: pricing, SaaS, business model
  entities: ['K...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Report on Emerging AI Tools for Automating Competitive Market Analysis                         │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  This report evaluates five emerging AI tools designed to automate competitive market analysis, focusing on     │
│  Klue, Crayon, Semrush, Similarweb, and Competely. Each tool's unique features, limitations, cost structures,   │
│  and ideal use cases are analyzed. The findings reveal that while these tools can significantly enhance         │
│  competitive intelligence for mid-sized marketing firms, each comes with specific strengths and weaknesses      │
│  that organizations must consider based on their operational needs and budget constraints.                      │
│                                                                                                                 │
│  ## Detailed Findings                                                                                           │
│                                                                                                                 │
│  ### 1. Klue                                                                                                    │
│  - **Overview**: Klue is a competitive enablement platform that aggregates competitive intelligence to inform   │
│  sales and marketing strategies.                                                                                │
│  - **Features**:                                                                                                │
│    - **Compete Agent**: Automates the collection and sharing of competitive insights.                           │
│    - **Win/Loss Analysis**: Gathers customer feedback to refine go-to-market strategies.                        │
│    - **Integrations**: Works with Salesforce, Slack, and Teams, allowing for seamless workflow integration.     │
│    - **Dynamic Battlecards**: Offers real-time updates on competitors (source:                                  │
│  [Zendikt](https://www.zendikt.com/product/klue)).                                                              │
│  - **Limitations**:                                                                                             │
│    - High costs, typically ranging from **$15,000 to $40,000 annually**, with entry starting around             │
│  **$20,000**.                                                                                                   │
│    - Requires dedicated personnel to manage the database (source: [Klue User Feedback](https://www.klue.com)).  │
│  - **Ideal Use Case**: Best suited for large B2B sales teams that require continuous updates and competitive    │
│  insights to drive effectiveness during sales calls.                                                            │
│                                                                                                                 │
│  ### 2. Crayon                                                                                                  │
│  - **Overview**: Crayon offers a comprehensive competitive intelligence platform that monitors competitors and  │
│  delivers insightful analytics.                        

Report successfully saved to: research_report-p2.md


/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/pydantic/main.py:250: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x12a5694e0>
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/pydantic/main.py:250: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x12a568b80>
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/encoding_flow.py:379: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/Users/olaoluadisa/Documents/W

In [11]:
from IPython.display import Markdown, display

#Read the markdown file
with open("research_report-p2.md", "r") as f:
    content = f.read()

#Display it as a rendered markdown
display(Markdown(content))

# Comprehensive Report on Emerging AI Tools for Automating Competitive Market Analysis

## Executive Summary
This report evaluates five emerging AI tools designed to automate competitive market analysis, focusing on Klue, Crayon, Semrush, Similarweb, and Competely. Each tool's unique features, limitations, cost structures, and ideal use cases are analyzed. The findings reveal that while these tools can significantly enhance competitive intelligence for mid-sized marketing firms, each comes with specific strengths and weaknesses that organizations must consider based on their operational needs and budget constraints.

## Detailed Findings

### 1. Klue
- **Overview**: Klue is a competitive enablement platform that aggregates competitive intelligence to inform sales and marketing strategies.
- **Features**:
  - **Compete Agent**: Automates the collection and sharing of competitive insights.
  - **Win/Loss Analysis**: Gathers customer feedback to refine go-to-market strategies.
  - **Integrations**: Works with Salesforce, Slack, and Teams, allowing for seamless workflow integration.
  - **Dynamic Battlecards**: Offers real-time updates on competitors (source: [Zendikt](https://www.zendikt.com/product/klue)).
- **Limitations**:
  - High costs, typically ranging from **$15,000 to $40,000 annually**, with entry starting around **$20,000**.
  - Requires dedicated personnel to manage the database (source: [Klue User Feedback](https://www.klue.com)).
- **Ideal Use Case**: Best suited for large B2B sales teams that require continuous updates and competitive insights to drive effectiveness during sales calls.

### 2. Crayon
- **Overview**: Crayon offers a comprehensive competitive intelligence platform that monitors competitors and delivers insightful analytics.
- **Features**:
  - **Automated Monitoring**: Tracks competitors across various digital platforms, providing actionable insights.
  - **Reports**: Automatically generates competitor reports prioritizing significant changes using AI-driven insights (source: [Crayon](https://www.crayon.co)).
  - **Battlecards**: Tailors competitive battlecards for sales teams.
- **Limitations**:
  - Pricing starts around **$15,000/year**, with the average contract value being about **$29,500**.
  - May require a dedicated competitive intelligence function and lacks a free tier (source: [Vendr](https://www.vendr.com/marketplace/crayon)).
- **Ideal Use Case**: Particularly valuable for mid-sized to enterprise-level sales teams facing rapid competitive changes, especially those utilizing Salesforce.

### 3. Semrush
- **Overview**: Semrush is predominantly known for its digital marketing capabilities, expanding into AI-driven insights that drive market visibility.
- **Features**:
  - **AI Visibility Toolkit**: Monitors brand presence across AI-driven platforms.
  - **Comprehensive SEO Tools**: Encompasses keyword tracking, backlink analysis, and competitor traffic insights (source: [Semrush](https://www.semrush.com/kb/1493-ai-visibility-toolkit)).
- **Limitations**:
  - Pricing ranges from **$99 to $249/month**, which can escalate with the addition of features.
  - Complexity arises for users solely focused on AI monitoring without the necessity for SEO, potentially overwhelming smaller firms (source: [ContentForce](https://blog.contentforce.ai/similarweb-pricing/)).
- **Ideal Use Case**: Optimal for small to mid-sized businesses eager for an all-in-one solution that enhances both SEO and AI visibility strategies.

### 4. Similarweb
- **Overview**: Similarweb provides market intelligence analytical tools vital for understanding competitors and traffic dynamics.
- **Features**:
  - **Traffic Source Analysis**: Offers insights into visitor behavior and competitive benchmarks.
  - **AI Search Intelligence**: Monitors performance across AI-driven platforms and provides competitive analytics (source: [Similarweb](https://www.similarweb.com)).
- **Limitations**:
  - Pricing starts at **$125/month**. Many advanced features require higher-tier plans, which makes it less feasible for smaller organizations (source: [GetApp](https://www.getapp.com/business-intelligence-analytics-software/a/similarweb-pro/)).
- **Ideal Use Case**: Best for teams looking to understand extensive market insights and traffic benchmarks beyond basic SEO functionalities.

### 5. Competely
- **Overview**: Competely utilizes AI to facilitate quick, actionable competitor analysis.
- **Features**:
  - **Instant Competitive Analysis**: Provides succinct summaries and structured comparisons across various competitors.
  - **Monitoring Briefs**: Sends alerts to users regarding significant competitor changes (source: [Competely](https://competely.ai)).
- **Limitations**:
  - Subscription prices range from **$39 to $99/month**, primarily relying on available public data, which may lack depth for niche competitors.
  - Pricing structure could be less transparent, leading to potential confusion over actual costs (source: [Competely Overview](https://competely.ai)).
- **Ideal Use Case**: Aimed at startups and small to mid-sized companies that need fast and effective competitive insights for strategy adjustments.

## Supporting Evidence and Analysis
The analysis was conducted by comparing the reported capabilities, limitations, and pricing structures of the five tools using verified data sources. The outlined features and limitations emphasize the practical implications for mid-sized marketing firms (source reliability rates: high, medium).

## Complete Source Citations
1. Klue. (n.d.). Retrieved from [Zendikt](https://www.zendikt.com/product/klue)
2. Crayon. (n.d.). Retrieved from [Crayon](https://www.crayon.co)
3. Semrush. (n.d.). Retrieved from [Semrush](https://www.semrush.com/kb/1493-ai-visibility-toolkit)
4. Similarweb. (n.d.). Retrieved from [Similarweb](https://www.similarweb.com)
5. Competely. (n.d.). Retrieved from [Competely](https://competely.ai)

## Key Insights and Recommendations
1. **Choosing the Right Tool**: Firms should evaluate their specific needs, particularly engagement levels with competitors and the depth of analysis required before making a selection.
2. **Budget Considerations**: Price sensitivity is crucial; tools like Competely provide lower entry points for quick insights while higher-priced options like Klue offer comprehensive systems suited for larger teams.
3. **Integration and Usability**: Consider the learning curve and integration aspects when selecting tools to ensure teams can utilize features effectively without extensive training.

## Conclusion
The five evaluated AI tools present diverse functionalities for competitive market analysis. Mid-sized firms should weigh their operational requirements against each tool's strengths, limitations, and costs to make informed decisions that align with their strategic objectives. By leveraging these emerging AI tools effectively, firms can significantly enhance their ability to track competition and adjust their strategies accordingly.